# Livrable 3 – Génération de légendes d’images (Image captioning)

## Objectif

Ce livrable constitue la dernière étape du pipeline du projet **Leyanda**. L’objectif est de concevoir un modèle de deep learning capable de **générer automatiquement une légende textuelle pour une photographie**, en s'appuyant sur le dataset **MS COCO**.

Le modèle est composé de deux sous-parties :
- Un **réseau de neurones convolutifs (CNN)** pour encoder l’image en une représentation vectorielle.
- Un **réseau de neurones récurrents (RNN)** qui génère une phrase mot à mot à partir de cette représentation.

---

## Pipeline général

![Worklofw](../assets/Soutenance/workflow.png)

---

## Architecture du modèle de Captioning

### 1. Encodeur (CNN)

- **Type de réseau utilisé** : InceptionV3 pré-entraîné sur ImageNet.
- Rôle : extraire des **features visuels** à partir de l’image.
- Les couches supérieures du modèle sont supprimées pour ne conserver que les **vecteurs d’activation**.

```python
from tensorflow.keras.applications import InceptionV3

# Chargement du modèle pré-entraîné
cnn_model = InceptionV3(weights='imagenet', include_top=False)```


# Exemple d'architecture (simplifiée)
```python
image_features = Input(shape=(2048,))
image_embedding = Dense(256, activation='relu')(image_features)

text_input = Input(shape=(max_seq_length,))
text_embedding = Embedding(vocab_size, 256, mask_zero=True)(text_input)
lstm_out = LSTM(512)(text_embedding)

merged = add([image_embedding, lstm_out])
output = Dense(vocab_size, activation='softmax')(merged)
```


## 2. Décodeur (RNN)

Type de réseau utilisé : LSTM avec embedding et couche dense en sortie.

Rôle : générer la séquence textuelle en sortie à partir du vecteur image.

```python
# Exemple d'architecture (simplifiée)
image_features = Input(shape=(2048,))
image_embedding = Dense(256, activation='relu')(image_features)

text_input = Input(shape=(max_seq_length,))
text_embedding = Embedding(vocab_size, 256, mask_zero=True)(text_input)
lstm_out = LSTM(512)(text_embedding)

merged = add([image_embedding, lstm_out])
output = Dense(vocab_size, activation='softmax')(merged)

```

# Prétraitement
Images
- Redimensionnement et mise à l’échelle des images.
- Passage dans un CNN pour extraire les features d’encodage.
- (Optionnel) Denoising ou sharpening en amont de l’encodage.

Textes
- Nettoyage des légendes.
- Vectorisation en tokens via un tokenizer personnalisé.
- Padding des séquences.

## Performances du modèle

L'entraînement du modèle est suivi à l’aide de l’objet `history`, qui enregistre la perte (`loss`) pour l'ensemble d'entraînement et de validation.

```python
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.title("Courbes d'apprentissage")
plt.xlabel("Époques")
plt.ylabel("Loss")
```

Ces courbes permettent de visualiser l’évolution de la fonction de perte au fil des époques, et ainsi d’évaluer la stabilité et la convergence du modèle.

⚠️ Une tentative d'intégration de la métrique BLEU pendant l'entraînement a été faite. Toutefois, le calcul du score BLEU à chaque époque ralentissait significativement l'entraînement, en particulier à cause du volume important de données de validation et du coût de génération des séquences complètes.

Nous avons donc déplacé le calcul du BLEU score à la phase d'inférence, afin d’évaluer les performances du modèle uniquement sur un sous-ensemble d’images tests après entraînement.